In [30]:
import pandas as pd
import os
from itertools import permutations, combinations
import re

## Clorofila de las boyas

Funciones para hacer la media de un cierto rango de profundidades y para crear un único dataframe con todas las boyas, sus localizaciones y la clorofila a partir del diccionario de dataframes.

In [41]:
def add_average_column(df_dict, depth):
    for name, df in df_dict.items():
        df = df.copy()  
        
        columns = ['0.0', '0.5', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5', '4.0', '4.5', '5.0']
        
        # Select columns to average (excluding '0.5' and '1.0')
        if depth == ">1":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['0.0', '0.5', '1.0']]
        if depth == "<1":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['1.5','2.0','2.5', '3.0', '3.5', '4.0', '4.5', '5.0']]
        if depth == "<2":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['2.5', '3.0', '3.5', '4.0', '4.5', '5.0']]
        if depth == "=1":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['1.0']]
        if depth == "=0":
            cols_to_avg = [col for col in columns if col in df.columns and col in ['0.0', '0.5']]

        # Compute the mean and add a new column
        if cols_to_avg:
            df['Average'] = df[cols_to_avg].mean(axis=1).round(3)
    
        # Update the dictionary with the modified DataFrame
        df_dict[name] = df

    return df_dict


def create_combined_dataframe(df_dict):
    data = []
    
    for name in sorted(df_dict.keys()):
        if name.startswith("CTD5") or name.startswith("CTD9"):
            continue  # Ignore CTD5 and CTD9
        
        buoy_name = name.split('_')[0]  # Extract buoy name (CTD1, CTD2, etc.)
        df = df_dict[name]
        
        if 'Average' in df.columns:
            for _, row in df.iterrows():
                data.append({'Date': row['Date'], 'Buoy': buoy_name, 'Chl': row['Average']})
    
    combined_df = pd.DataFrame(data).sort_values(by=["Date", "Buoy"])
    combined_df["Buoy"] = combined_df["Buoy"].str.replace(r"CTD-E", "CTD", regex=True)
    return combined_df

#### UPCT

In [42]:
path = "boyaUPCT"
filename = "locBoyasUPCT_reproyectado.csv"
loc_boyas = pd.read_csv(os.path.join(path, filename)).iloc[:,:5]

In [43]:
path = "boyaUPCT/extractedData/"
buoy_ids = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
dataframes_boyas = {}
for buoy_id in buoy_ids:
    archivos_csv = [f for f in os.listdir(path + buoy_id) if f.endswith('.csv')]
    for archivo in archivos_csv:     
        nombre_variable = os.path.splitext(archivo)[0]
        ruta_completa = os.path.join(path, buoy_id, archivo)
        if nombre_variable in ["Clorofila"]:
            dataframes_boyas[f"{buoy_id}_{nombre_variable}"] = pd.read_csv(ruta_completa)
for key, df in dataframes_boyas.items():
    df['Date'] = pd.to_datetime(df['Date'])

In [44]:
depth = ">1" # >1, <1, <2, =0, =1
depth_names = {"=0": "eq_0", "=1" : "eq_1", "<1" : "lt_1", "<2" : "lt_2", ">1": "gt_1"}
dataframes_boyas = add_average_column(dataframes_boyas, depth)
df_boyas = create_combined_dataframe(dataframes_boyas)

In [45]:
df_boyas.head(2)

,Date,Buoy,Chl
1119,2017-05-19,CTD1,0.883
0,2017-05-19,CTD10,0.837


In [29]:
df_boyas.to_csv(f"saved_files/df_boyas_upct_depth_{depth_names[depth]}.csv", index=False)

#### IMIDA

In [67]:

# Ruta a la carpeta que contiene todas las boyas
path = "boyasProf/"
# Listamos los nombres de todas las boyas
buoy_ids = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
# Diccionario para guardar dataframes
dataframes_boyas_imida = {}
# Para cada boya cargamos todos sus csvs y los guardamos con key f"{bouy_id}_{variable}"
for buoy_id in buoy_ids:
    archivos_csv = [f for f in os.listdir(path + buoy_id) if f.endswith('.csv')]
    for archivo in archivos_csv:     
        nombre_variable = os.path.splitext(archivo)[0]
        ruta_completa = os.path.join(path, buoy_id, archivo)
        if nombre_variable in ["Clorofila"]:
            dataframes_boyas_imida[f"{buoy_id}_{nombre_variable}"] = pd.read_csv(ruta_completa)
for key, df in dataframes_boyas_imida.items():
    df['Date'] = pd.to_datetime(df['fecha'])

In [68]:
for key, df in dataframes_boyas_imida.items():
    nuevo_nombre_columnas = {}
    for col in df.columns:
        match = re.search(r'profundidad_(-?\d+\.?\d*)', col)
        if match:
            nuevo_nombre_columnas[col] = str(abs(float(match.group(1))))
    dataframes_boyas_imida[key].rename(columns=nuevo_nombre_columnas, inplace=True)

# Lista de profundidades deseadas como strings
cols_to_keep = ['Date', '0.0', '1.0', '2.0', '3.0', '4.0', '5.0']

for key, df in dataframes_boyas_imida.items():
    columnas_filtradas = [col for col in df.columns if col in cols_to_keep]
    dataframes_boyas_imida[key] = df[columnas_filtradas]

In [69]:
depth = ">1" # >1, <1, <2, =0, =1
depth_names = {"=0": "eq_0", "=1" : "eq_1", "<1" : "lt_1", "<2" : "lt_2", ">1": "gt_1"}
dataframes_boyas_imida = add_average_column(dataframes_boyas_imida, depth)
df_boyas = create_combined_dataframe(dataframes_boyas_imida)

In [70]:
df_boyas.head(2)

,Date,Buoy,Chl
3550,2016-06-08,CTD8,12.62
2131,2016-06-29,CTD4,11.35


In [71]:
df_boyas.to_csv(f"saved_files/df_boyas_imida_depth_{depth_names[depth]}.csv", index=False)

## Unión de clorofila y reflectancias

In [72]:
# Cargamos los csv de los tifs
path = "saved_files/"
dfs_tifs = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_tiffs_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_tifs[nombre_sin_extension] = pd.read_csv(ruta_completa)

# Y de las boyas
path = "saved_files/"
dfs_boyas = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_boyas_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_boyas[nombre_sin_extension] = pd.read_csv(ruta_completa)

In [73]:
band_names = {
    "Band_1": "rhow_B1",
    "Band_2": "rhow_B2",
    "Band_3": "rhow_B3",
    "Band_4": "rhow_B4",
    "Band_5": "rhow_B5",
    "Band_6": "rhow_B6",
    "Band_7": "rhow_B7",
    "Band_8": "rhow_B8",
    "Band_9": "rhown_B1",
    "Band_10": "rhown_B2",
    "Band_11": "rhown_B3",
    "Band_12": "rhown_B4",
    "Band_13": "rhown_B5",
    "Band_14": "rhown_B6",
}
for nombre_df, df in dfs_tifs.items():
    dfs_tifs[nombre_df] = df.rename(columns=band_names)


In [74]:
dfs_tifs["df_tiffs_c2x-complex-nets_1x1"].head(3)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,rhow_B7,rhow_B8,rhown_B1,rhown_B2,rhown_B3,rhown_B4,rhown_B5,rhown_B6,Band_15,Band_16
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.002320,0.002351,0.000930,0.023582,0.034339,0.043720,0.015355,0.009758,0.002110,0.001174,-8.000000e-45
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.001000,0.001059,0.000427,0.014022,0.022509,0.026334,0.007046,0.004299,0.000942,0.000236,-8.000000e-45
2,2017-06-30,CTD3,4181698,695238,0.017592,0.024643,0.031816,0.010574,0.007355,0.001946,0.001922,0.000778,0.017462,0.024656,0.031839,0.010682,0.007440,0.001862,0.104799,-8.000000e-45


In [75]:
dfs_boyas["df_boyas_upct_depth_gt_1"].head(3)

,Date,Buoy,Chl
0,2017-05-19,CTD1,0.883
1,2017-05-19,CTD10,0.837
2,2017-05-19,CTD11,0.899


In [82]:
merge_dict = {}

for df_tif_name, df_tif in dfs_tifs.items():
    for df_boya_name, df_boya in dfs_boyas.items():
        print(df_tif_name[9:], df_boya_name[9:])
        merge_dict[f"{df_tif_name[9:]}_{df_boya_name[9:]}"] = df_tif.merge(df_boya, how="inner", on=["Date", "Buoy"])    

c2x-nets_5x5 upct_depth_gt_1
c2x-nets_5x5 imida_depth_gt_1
c2x-nets_5x5 imida_depth_eq_1
c2x-nets_5x5 upct_depth_eq_1
c2x-nets_5x5 imida_depth_lt_2
c2x-nets_5x5 imida_depth_eq_0
c2x-nets_5x5 upct_depth_lt_2
c2x-nets_5x5 upct_depth_lt_1
c2x-nets_5x5 imida_depth_lt_1
c2x-nets_5x5 upct_depth_eq_0
c2x-nets_3x3 upct_depth_gt_1
c2x-nets_3x3 imida_depth_gt_1
c2x-nets_3x3 imida_depth_eq_1
c2x-nets_3x3 upct_depth_eq_1
c2x-nets_3x3 imida_depth_lt_2
c2x-nets_3x3 imida_depth_eq_0
c2x-nets_3x3 upct_depth_lt_2
c2x-nets_3x3 upct_depth_lt_1
c2x-nets_3x3 imida_depth_lt_1
c2x-nets_3x3 upct_depth_eq_0
c2x-complex-nets_5x5 upct_depth_gt_1
c2x-complex-nets_5x5 imida_depth_gt_1
c2x-complex-nets_5x5 imida_depth_eq_1
c2x-complex-nets_5x5 upct_depth_eq_1
c2x-complex-nets_5x5 imida_depth_lt_2
c2x-complex-nets_5x5 imida_depth_eq_0
c2x-complex-nets_5x5 upct_depth_lt_2
c2x-complex-nets_5x5 upct_depth_lt_1
c2x-complex-nets_5x5 imida_depth_lt_1
c2x-complex-nets_5x5 upct_depth_eq_0
c2x-complex-nets_1x1 upct_depth_gt_

In [83]:
for df_name, df in merge_dict.items():
    df.to_csv(f"saved_files/dataset/{df_name}.csv", index=False)

## Creación de features

In [89]:
# Cargamos los csv de los tifs
path = "saved_files/dataset/"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith(".csv") and not archivo.endswith("_features.csv") :
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)

Fórmulas a utilizar:
- Diferencia normalizada $$\frac{R(\lambda_1) - R(\lambda_2)}{R(\lambda_1) + R(\lambda_2)}$$
- Dall-Gitelson $$\left(\frac{1}{R(\lambda_1)}-\frac{1}{R(\lambda_2)}\right) \times R(\lambda_3)$$
- Diferencia normalizada 4 bandas $$\frac{R(\lambda_1) - R(\lambda_2)}{R(\lambda_3) + R(\lambda_4)}$$
- Diferencia inversas $$\frac{1}{R(\lambda_1)}-\frac{1}{R(\lambda_2)}$$
- Diferencia relación 4 bands $$\frac{R(\lambda_1)}{R(\lambda_2)}-\frac{R(\lambda_3)}{R(\lambda_4)}$$

In [90]:
def diferencia_normalizada(band1, band2):
    value = (band1 - band2)/(band1 + band2)
    return value.round(3)

def dall_gitelson(band1, band2, band3):
    value = (1/(band1) - 1/(band2))*(band3)
    return value.round(3)

def diferencia_normalizada_4bandas(band1, band2, band3, band4):
    value = (band1 - band2)/(band3 + band4)
    return value.round(3)

def diferencia_inversas(band1, band2):
    value = 1/(band1) - 1/(band2)
    return value.round(3)

def diferencia_relacion_4bandas(band1, band2, band3, band4):
    value = band1/band2 - band3/band4
    return value.round(3)

Añadimos la diferencia normalizada y diferencia de inversas para las bandas de Ultra Blue, Blue, Green , Red, NIR1; lo hacemos dos veces, con rhow y con rhown.

In [91]:
index_list = []
def add_two_band_difs(data, bands):
    for i, band1 in enumerate(bands):
        for band2 in bands[i+1:]:
            colname_dif_norm = f"dif_norm_{band1}_{band2}"
            data[colname_dif_norm] = diferencia_normalizada(data[band1], data[band2])
            index_list.append(colname_dif_norm)
            colname_dif_inv = f"dif_inv_{band1}_{band2}"
            data[colname_dif_inv] = diferencia_inversas(data[band1], data[band2])
            index_list.append(colname_dif_inv)
    return data


Añadimos la relación de Dall-Gitelson, evitando repeticiones por la simetría de $f(b1,b2,b3)=−f(b2,b1,b3)$

In [92]:
index_dall_gitelson_list = []
def add_dall_gitelson(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita repeticiones de pares
        for band3 in bands:
            if band3 not in (band1, band2):  # evitar que band3 sea igual a los anteriores
                colname = f"dall_gitelson_{band1}_{band2}_{band3}"
                data[colname] = dall_gitelson(data[band1], data[band2], data[band3])
                index_dall_gitelson_list.append(colname)
    return data


Añadimos el índice de tipo diferencia normalizada que utiliza 4 bands, forzando a que estas 4 sean diferentes y evitando redundancia por las simetrías $(b1−b2)/(b3+b4)=(b1−b2)/(b4+b3)$ y $(b1−b2)/(b3+b4)=−(b2−b1)/(b3+b4)$

In [93]:
index_dif_norm_4bands_list = []
def add_norm_dif_4bands(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita invertir band1 y band2
        for band3, band4 in combinations(bands, 2):  # evita invertir band3 y band4
            # Asegurar que todas las bandas son distintas
            if len({band1, band2, band3, band4}) == 4:
                colname = f"dif_norm_4_bands_{band1}_{band2}_{band3}_{band4}"
                data[colname] = diferencia_normalizada_4bandas(
                    data[band1], data[band2], data[band3], data[band4]
                )
                index_dif_norm_4bands_list.append(colname)
    return data

Añadimos el cociente entre dos parejas de bandas, también evitando simetría, en este caso:

$\frac{b1}{b2} - \frac{b3}{b4} = -\left(\frac{b3}{b4} - \frac{b1}{b2} \right)$

In [94]:
index_dif_rel_4bands_list = []

def add_index_dif_rel_4bands(data, bands):
    for band1, band2, band3, band4 in permutations(bands, 4):
        # Evitar redundancias por simetría de términos
        # Criterio: solo aceptamos combinaciones donde el primer término es "menor" que el segundo
        if (band1, band2) < (band3, band4):  # evita generar la versión espejo con signo opuesto
            colname = f"dif_rel_4bands_{band1}_{band2}_{band3}_{band4}"
            data[colname] = diferencia_relacion_4bandas(
                data[band1], data[band2], data[band3], data[band4]
            )
            index_dif_rel_4bands_list.append(colname)
    return data

Aplicamos todas las fórmulas anteriores sobre los dataframes del diccionario de dataframes, para los conjuntos de bandas de rhow y rhown.

In [95]:
band_sets = [
    ['rhow_B1', 'rhow_B2', 'rhow_B3', 'rhow_B4', 'rhow_B5'],
    ['rhown_B1', 'rhown_B2', 'rhown_B3', 'rhown_B4', 'rhown_B5']
]
for nombre_df, df in dfs.items():
    for bands_to_use in band_sets:
        df = add_two_band_difs(df, bands_to_use)
        df = add_dall_gitelson(df, bands_to_use)
        df = add_norm_dif_4bands(df, bands_to_use)
        df = add_index_dif_rel_4bands(df, bands_to_use)
    dfs[nombre_df] = df 

/tmp/ipykernel_27738/3378284678.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[colname] = diferencia_relacion_4bandas(
/tmp/ipykernel_27738/3378284678.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[colname] = diferencia_relacion_4bandas(
/tmp/ipykernel_27738/3378284678.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmente

Ya tenemos todos los dataframes con todas las columnas.

In [96]:
dfs["c2x-complex-nets_1x1_imida_depth_gt_1"].head(2)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhown_B3_rhown_B4_rhown_B5_rhown_B1,dif_rel_4bands_rhown_B3_rhown_B4_rhown_B5_rhown_B2,dif_rel_4bands_rhown_B3_rhown_B5_rhown_B4_rhown_B1,dif_rel_4bands_rhown_B3_rhown_B5_rhown_B4_rhown_B2,dif_rel_4bands_rhown_B4_rhown_B1_rhown_B5_rhown_B2,dif_rel_4bands_rhown_B4_rhown_B1_rhown_B5_rhown_B3,dif_rel_4bands_rhown_B4_rhown_B2_rhown_B5_rhown_B1,dif_rel_4bands_rhown_B4_rhown_B2_rhown_B5_rhown_B3,dif_rel_4bands_rhown_B4_rhown_B3_rhown_B5_rhown_B1,dif_rel_4bands_rhown_B4_rhown_B3_rhown_B5_rhown_B2
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.00232,...,2.434,2.563,3.829,4.033,0.367,0.428,0.033,0.224,-0.063,0.067
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.00100,...,3.431,3.546,5.623,5.812,0.311,0.339,0.006,0.150,-0.039,0.077


Renombramos columnas para que los nombres no incluyan varias veces rhow / rhown. Es decir, que index_diff_ratio_rhow_B1_rhow_B4_rhow_B2_rhow_B5 sea solamente index_diff_ratio_rhow_B1_B4_B2_B5.

In [97]:
import re

def compactar_prefijos_columnas(df):
    nuevo_nombre_columnas = {}

    for col in df.columns:
        # Detectar columnas con patrones tipo index_algo_rhow_B1_rhow_B2_...
        if re.search(r'(rhow|rhown)(_B\d+)+', col):
            partes = col.split('_')
            base = []
            bandas = []
            prefijo = None

            for parte in partes:
                if parte in ['rhow', 'rhown']:
                    if not prefijo:
                        prefijo = parte
                elif parte.startswith('B'):
                    bandas.append(parte)
                else:
                    base.append(parte)

            if prefijo and bandas:
                #nuevo_nombre = f"{'_'.join(base)}_{prefijo}_{'_'.join(bandas)}"
                if base:
                    nuevo_nombre = f"{'_'.join(base)}_{prefijo}_{'_'.join(bandas)}"
                else:
                    nuevo_nombre = f"{prefijo}_{'_'.join(bandas)}"

                nuevo_nombre_columnas[col] = nuevo_nombre

    # Renombrar columnas
    df = df.rename(columns=nuevo_nombre_columnas)
    return df


In [98]:
for nombre_df, df in dfs.items():
    dfs[nombre_df] = compactar_prefijos_columnas(df)

In [99]:
dfs["c2x-complex-nets_1x1_imida_depth_gt_1"].head(2)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhown_B3_B4_B5_B1,dif_rel_4bands_rhown_B3_B4_B5_B2,dif_rel_4bands_rhown_B3_B5_B4_B1,dif_rel_4bands_rhown_B3_B5_B4_B2,dif_rel_4bands_rhown_B4_B1_B5_B2,dif_rel_4bands_rhown_B4_B1_B5_B3,dif_rel_4bands_rhown_B4_B2_B5_B1,dif_rel_4bands_rhown_B4_B2_B5_B3,dif_rel_4bands_rhown_B4_B3_B5_B1,dif_rel_4bands_rhown_B4_B3_B5_B2
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.00232,...,2.434,2.563,3.829,4.033,0.367,0.428,0.033,0.224,-0.063,0.067
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.00100,...,3.431,3.546,5.623,5.812,0.311,0.339,0.006,0.150,-0.039,0.077


Guardamos los dataframes como csvs, con el mismo nombre que tenían pero añadiendo "_features" al final.

In [100]:
for df_name, df in dfs.items():
    df.to_csv(f"saved_files/dataset/{df_name}_features.csv", index=False)